[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/anymap-ts/blob/main/docs/maplibre/geoagent.ipynb)
[![Open in Notebook.link](https://img.shields.io/badge/notebook-link-e2d610?logo=jupyter&logoColor=white)](https://notebook.link/opengeos/anymap-ts/blob/main/docs/maplibre/geoagent.ipynb)

# GeoAgent Control

This notebook demonstrates the `maplibre-gl-geoagent` control in `anymap-ts`. The control runs in the browser, stores provider settings in browser session storage, and can inspect or operate on the live MapLibre map through browser map tools.

In [ ]:
# %pip install anymap-ts

## Basic GeoAgent Control

The browser panel lets users choose a provider, enter an API key, and select a model. API keys are not passed through Python; they are entered and stored in the browser session.

In [ ]:
from anymap_ts import Map

m = Map(center=[-122.431, 37.789], zoom=12, height="720px")
m.add_geoagent_control(
    position="top-left",
    collapsed=True,
    panel_width=420,
    allow_code_execution_default=True,
    allow_destructive_tools_default=True,
    show_permission_toggles=False,
)
m

## Add Map Data for the Agent

Add a few visible layers before opening the assistant. The agent can inspect layer names, query rendered features, move the camera, change basemaps, and create map annotations.

In [ ]:
from anymap_ts import Map

landmarks = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [-122.4194, 37.7749]},
            "properties": {"name": "San Francisco City Hall", "category": "civic"},
        },
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [-122.3937, 37.7955]},
            "properties": {"name": "Ferry Building", "category": "landmark"},
        },
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [-122.4783, 37.8199]},
            "properties": {"name": "Golden Gate Bridge", "category": "landmark"},
        },
    ],
}

sample_area = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [-122.431, 37.789],
                        [-122.402, 37.789],
                        [-122.402, 37.773],
                        [-122.431, 37.773],
                        [-122.431, 37.789],
                    ]
                ],
            },
            "properties": {"name": "Central SF sample area"},
        }
    ],
}

m = Map(center=[-122.431, 37.789], zoom=12, height="720px")
m.add_geojson(
    sample_area,
    name="sf-sample-area",
    layer_type="fill",
    paint={"fill-color": "#2f9e44", "fill-opacity": 0.25},
)
m.add_geojson(
    landmarks,
    name="sf-landmarks",
    layer_type="circle",
    paint={
        "circle-color": "#0b7285",
        "circle-radius": 8,
        "circle-stroke-color": "#ffffff",
        "circle-stroke-width": 2,
    },
)
m.add_geoagent_control(
    position="top-left",
    collapsed=False,
    panel_width=420,
    default_provider="openai-responses",
    allow_code_execution_default=False,
    allow_destructive_tools_default=False,
    show_permission_toggles=True,
    basemaps={
        "Liberty": "https://tiles.openfreemap.org/styles/liberty",
        "Positron": "https://basemaps.cartocdn.com/gl/positron-gl-style/style.json",
        "DarkMatter": "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
    },
)
m

## GeoAgent with Google Earth Engine

The GeoAgent control can expose Google Earth Engine catalog and layer tools. In Jupyter, authenticate and initialize the Earth Engine Python API first. `add_geoagent_control()` will pass a short-lived Python-side Earth Engine access token to the browser control, avoiding Google browser OAuth inside notebook widget output.

If the Python Earth Engine API is not initialized, the control can still use `GEE_OAUTH_CLIENT_ID` and `GEE_PROJECT_ID` from the Python environment and fall back to browser OAuth in a top-level browser page.

In [ ]:
# Uncomment and run once if Earth Engine is not already initialized.
# %pip install earthengine-api

In [ ]:
import ee

ee.Authenticate()
ee.Initialize(project="your-ee-project")

In [ ]:
from anymap_ts import Map

knoxville_area = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [-84.18, 35.82],
                        [-83.66, 35.82],
                        [-83.66, 36.11],
                        [-84.18, 36.11],
                        [-84.18, 35.82],
                    ]
                ],
            },
            "properties": {"name": "Knoxville study area"},
        }
    ],
}

m = Map(center=[-83.9207, 35.9606], zoom=8, height="720px")
m.add_geojson(
    knoxville_area,
    name="knoxville-study-area",
    layer_type="fill",
    paint={"fill-color": "#2f9e44", "fill-opacity": 0.22},
)
m.add_geoagent_control(
    position="top-left",
    collapsed=False,
    title="GeoAgent GEE",
    panel_width=460,
    default_provider="google",
    storage_prefix="geoagent.maplibre.gee",
    allow_code_execution_default=False,
    allow_destructive_tools_default=False,
    show_permission_toggles=False,
    earth_engine={
        "enabled": True,
        "includeCommunityCatalog": True,
    },
)
m

## Read and Update GeoAgent State

The widget keeps the latest browser-side control state in `geoagent_state`. State updates are asynchronous, so run the state-reading cell after interacting with the panel.

In [ ]:
m.geoagent_state

In [ ]:
m.set_geoagent_state(collapsed=True)

## Remove the Control

In [ ]:
# m.remove_geoagent_control()